<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [1]:

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False)

    X = mnist.data
    y = mnist.target.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )

    return X_train, X_test, y_train, y_test


Sim, a normalização é necessária como as imagens são compostas por pixels que variam de 0 a 255, deixá-los "puros" pode dificultar muito o aprendizado do modelo.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [2]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed):
    if len(X_train.shape) > 2:
        X_train = X_train.reshape(X_train.shape[0], -1)
    rf_model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)

    rf_model.fit(X_train, y_train)

    return rf_model

def train_adaboost(X_train, y_train, seed):
    if len(X_train.shape) > 2:
        X_train = X_train.reshape(X_train.shape[0], -1)

    ada_model = AdaBoostClassifier(n_estimators=50, random_state=seed)

    ada_model.fit(X_train, y_train)

    return ada_model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [3]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    if len(X_test.shape) > 2:
        X_test_flat = X_test.reshape(X_test.shape[0], -1)
    else:
        X_test_flat = X_test

    y_pred = model.predict(X_test_flat)

    acc = accuracy_score(y_test, y_pred)

    return acc

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [4]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        print(f"Iniciando treino: Random Forest (Seed: {seed})...")
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        print(f"Iniciando treino: AdaBoost (Seed: {seed})...")
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    accuracy = evaluate(model, X_test, y_test)

    print(f"Pipeline finalizado. Acurácia do {model_type.upper()}: {accuracy:.2%}")
    return accuracy

run_pipeline()

Iniciando treino: Random Forest (Seed: 42)...
Pipeline finalizado. Acurácia do RF: 88.24%


0.8824285714285715

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**


- Não existe uma profundidade específica onde o overfitting começa, depende da complexidade do dataset. No Fashion MNIST, o overfitting fica claro por volta das camadas 12 a 15.

- Quando não é posto um limite na profundidade da árvore, ela tem permissão de ir o quão fundo ela quiser, o máximo que a árvore consegue aprofundar é até que todas as folhas sejam puras.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [5]:
from sklearn.metrics import classification_report, accuracy_score

def run_full_comparison(seed):
    X_train, X_test, y_train, y_test = load_data(seed)

    print("Treinando Random Forest...")
    rf_model = train_random_forest(X_train, y_train, seed)
    rf_preds = rf_model.predict(X_test.reshape(X_test.shape[0], -1))

    print("Treinando AdaBoost (isso pode levar um pouco mais de tempo)...")
    ada_model = train_adaboost(X_train, y_train, seed)
    ada_preds = ada_model.predict(X_test.reshape(X_test.shape[0], -1))

    models = [("Random Forest", y_test, rf_preds), ("AdaBoost", y_test, ada_preds)]

    for name, true, pred in models:
        print(f"\n{'='*30}")
        print(f" RESULTADOS: {name}")
        print(f"{'='*30}")
        print(f"Acurácia Geral: {accuracy_score(true, pred):.4f}")
        print("\nRelatório de Classificação:")
        print(classification_report(true, pred))

#teste
run_full_comparison(42)

Treinando Random Forest...
Treinando AdaBoost (isso pode levar um pouco mais de tempo)...

 RESULTADOS: Random Forest
Acurácia Geral: 0.8824

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1394
           1       0.99      0.96      0.98      1402
           2       0.79      0.82      0.81      1407
           3       0.88      0.92      0.90      1449
           4       0.77      0.84      0.80      1357
           5       0.97      0.96      0.96      1449
           6       0.74      0.59      0.66      1407
           7       0.93      0.94      0.94      1359
           8       0.96      0.98      0.97      1342
           9       0.95      0.96      0.95      1434

    accuracy                           0.88     14000
   macro avg       0.88      0.88      0.88     14000
weighted avg       0.88      0.88      0.88     14000


 RESULTADOS: AdaBoost
Acurácia Geral: 0.5376

Relatório de Classificaç

**Qual modelo apresentou melhor desempenho inicial**

- O Random Forest apresentou um desempenho inicial muito superior.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [10]:
run_full_comparison(42)
run_full_comparison(7)

Treinando Random Forest...
Treinando AdaBoost (isso pode levar um pouco mais de tempo)...

 RESULTADOS: Random Forest
Acurácia Geral: 0.8824

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1394
           1       0.99      0.96      0.98      1402
           2       0.79      0.82      0.81      1407
           3       0.88      0.92      0.90      1449
           4       0.77      0.84      0.80      1357
           5       0.97      0.96      0.96      1449
           6       0.74      0.59      0.66      1407
           7       0.93      0.94      0.94      1359
           8       0.96      0.98      0.97      1342
           9       0.95      0.96      0.95      1434

    accuracy                           0.88     14000
   macro avg       0.88      0.88      0.88     14000
weighted avg       0.88      0.88      0.88     14000


 RESULTADOS: AdaBoost
Acurácia Geral: 0.5376

Relatório de Classificaç

Sim, o experimento é 100% reprodutível.
A reprodutibilidade em Machine Learning não significa que o resultado deve ser o mesmo para qualquer semente, mas sim que o mesmo resultado deve ser obtido sempre que a mesma semente for utilizada.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [11]:
def check_overfitting(seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    model = train_random_forest(X_train, y_train, seed)

    acc_train = evaluate(model, X_train, y_train)
    acc_test = evaluate(model, X_test, y_test)

    print(f"Acurácia no Treino: {acc_train:.4f}")
    print(f"Acurácia no Teste: {acc_test:.4f}")
    print(f"Diferença (Gap): {acc_train - acc_test:.4f}")

check_overfitting(42)

Acurácia no Treino: 1.0000
Acurácia no Teste: 0.8824
Diferença (Gap): 0.1176


Sim, existe overfitting significativo. O overfitting é caracterizado por um modelo que apresenta um desempenho excelente nos dados de treinamento, mas não consegue manter o mesmo nível nos dados de teste. No caso do seu Random Forest, o modelo atingiu virtualmente 100% de acurácia no treino, enquanto no teste ficou em 88%. Esse "gap" de aproximadamente 12% indica que o modelo memorizou ruídos e detalhes específicos das imagens de treino que não são úteis para generalizar para novas peças de roupa.


O Random Forest tende a sofrer mais com esse tipo de overfitting "bruto".

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [12]:
def experiment_hyperparameters(seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    n_estimators_list = [10, 50, 100]
    results_rf = []
    results_ab = []

    for n in n_estimators_list:
        rf = RandomForestClassifier(n_estimators=n, random_state=seed, n_jobs=-1)
        rf.fit(X_train.reshape(X_train.shape[0], -1), y_train)
        results_rf.append(evaluate(rf, X_test, y_test))

        ab = AdaBoostClassifier(n_estimators=n, random_state=seed)
        ab.fit(X_train.reshape(X_train.shape[0], -1), y_train)
        results_ab.append(evaluate(ab, X_test, y_test))

    return n_estimators_list, results_rf, results_ab

n_list, rf_res, ab_res = experiment_hyperparameters(42)

print(f"{'n_estimators':<15} | {'RF Accuracy':<15} | {'AB Accuracy':<15}")
for i in range(len(n_list)):
    print(f"{n_list[i]:<15} | {rf_res[i]:<15.4f} | {ab_res[i]:<15.4f}")

n_estimators    | RF Accuracy     | AB Accuracy    
10              | 0.8626          | 0.2592         
50              | 0.8806          | 0.5376         
100             | 0.8824          | 0.5219         


Random Forest: O desempenho tende a estabilizar rapidamente. A diferença entre 50 e 100 árvores costuma ser pequena, pois o Random Forest trabalha reduzindo a variância através da média. Uma vez que a floresta atinge um número suficiente de árvores para "cobrir" as variações dos dados, adicionar mais estimadores traz retornos decrescentes.

AdaBoost: O desempenho apresenta mudanças mais perceptíveis, mas muitas vezes erráticas. Como cada novo estimador tenta corrigir o erro do anterior, o modelo é mais dependente do número de iterações para conseguir separar as classes complexas do Fashion MNIST.

O AdaBoost é o modelo mais sensível a mudanças no hiperparâmetro 'n_estimators'

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acurácia é suficiente para avaliar os modelos?

Não. A acurácia ignora o desempenho individual de cada classe. Como visto no AdaBoost, uma acurácia razoável pode esconder falhas totais em categorias específicas como "Casacos".

Métricas como Precisão, Recall e F1-Score são indispensáveis. Elas revelam se o modelo confunde peças similares, como Camisetas e Camisas, permitindo uma análise muito mais detalhada do erro.

2. Como você garante que o resultado não ocorreu por acaso?

Através do uso de sementes de aleatoriedade (random_state) em todas as funções de sorteio e treino. Isso elimina o caos estatístico e garante que o experimento produza sempre os mesmos números.

Além disso, a consistência é validada testando o modelo com diferentes sementes. Se o desempenho se mantém estável entre a seed 42 e a 7, confirma-se que o aprendizado é robusto e não fruto de sorte no sorteio dos dados.

3. Cite dois possíveis problemas metodológicos neste experimento.

O primeiro é o overfitting extremo do Random Forest, que atinge 100% no treino por falta de limite na profundidade das árvores. O segundo é a comparação desigual: o RF usa árvores complexas enquanto o AdaBoost usa "stumps" simples, tornando a disputa desequilibrada para o Fashion MNIST.

4. O pipeline implementado é confiável? Justifique.

Sim, pois é estruturalmente correto e modular. Ele separa rigorosamente os dados de treino e teste e utiliza controles de aleatoriedade, o que evita o vazamento de informações e garante resultados honestos.

Contudo, para uso real, ele precisaria de ajuste de hiperparâmetros. Embora confiável como código, os modelos ainda operam com configurações padrão que não extraem o potencial máximo de generalização para o dataset.